In [24]:
import pandas as pd
from pycaret.regression import *
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np

In [12]:
df = pd.read_csv("dataset/HousePrice.csv")

In [13]:
df.drop(columns="Address",inplace=True)

In [14]:
data = df.copy()

Q1 = data["Price"].quantile(0.25)
Q3 = data["Price"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

data["outlier"] = ((data["Price"] < lower_bound) | (data["Price"] > upper_bound))

data = data.loc[data["outlier"] == False]

In [15]:
data.head(3)

,Area,Room,Parking,Warehouse,Elevator,Price,Price(USD),outlier
0,63,1,True,True,True,1850000000,61666.67,False
1,60,1,True,True,True,1850000000,61666.67,False
2,79,2,True,True,True,550000000,18333.33,False


In [17]:
X = data[["Area","Room","Parking","Warehouse","Elevator"]]
y = data["Price"]


new_df = X.copy()
new_df["Price"] = y

In [22]:
reg = setup(data=new_df,target="Price",session_id=42,train_size=0.7,normalize=True)

,Description,Value
0,Session id,42
1,Target,Price
2,Target type,Regression
3,Original data shape,"(3169, 6)"
4,Transformed data shape,"(3169, 6)"
5,Transformed train set shape,"(2218, 6)"
6,Transformed test set shape,"(951, 6)"
7,Numeric features,2
8,Preprocess,True
9,Imputation type,simple


In [23]:
best_model = compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,1372919676.1568,3984246395350650880.0000,1993015890.1056,0.5132,0.6998,0.8510,0.1140
knn,K Neighbors Regressor,1465186121.2001,4460145158309603328.0000,2109967410.2332,0.4523,0.7080,0.8242,0.0330
rf,Random Forest Regressor,1605685775.4513,5106906905950076928.0000,2257569869.5102,0.3776,0.7560,0.9500,0.1260
gbr,Gradient Boosting Regressor,1607634904.6099,5170560024060464128.0000,2270601990.0558,0.3706,0.7619,0.9520,0.0710
dt,Decision Tree Regressor,1607485706.8724,5198250617680011264.0000,2277376554.4620,0.3667,0.7572,0.9438,0.0300
ada,AdaBoost Regressor,1784392324.5402,5530995276850592768.0000,2348308865.8406,0.3274,0.8404,1.2054,0.0400
llar,Lasso Least Angle Regression,1779660779.8378,5663885038248913920.0000,2376350208.6758,0.3122,0.9331,1.1203,0.0260
lar,Least Angle Regression,1779660780.0310,5663885038915277824.0000,2376350208.8058,0.3122,0.9331,1.1203,0.0260
ridge,Ridge Regression,1779599112.3620,5663857865851123712.0000,2376343127.0300,0.3122,0.9321,1.1201,0.0280
lasso,Lasso Regression,1779660779.8494,5663885038220051456.0000,2376350208.6695,0.3122,0.9331,1.1203,0.0320


C:\Users\Administrator\Desktop\projects\Projects\House Price Prediction\venv\Lib\site-packages\pycaret\internal\pycaret_experiment\supervised_experiment.py:339: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_cols, subset=["TT (Sec)"])


In [26]:
et_model = create_model("et")

# =========================
# 4. Train Predictions
# =========================

train_predictions = predict_model(
    et_model,
    data=get_config("X_train")
)

y_train = get_config("y_train")
y_train_pred = train_predictions["prediction_label"]


# =========================
# 5. Test Predictions
# =========================

test_predictions = predict_model(et_model)

y_test = test_predictions["Price"]
y_test_pred = test_predictions["prediction_label"]


# =========================
# 6. Train Metrics
# =========================

r2_train = r2_score(y_train, y_train_pred)
mae_train = mean_absolute_error(y_train, y_train_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)


# =========================
# 7. Test Metrics
# =========================

r2_test = r2_score(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)


# =========================
# 8. R² Gap
# =========================

r2_gap = r2_train - r2_test


# =========================
# 9. Results
# =========================

print("=" * 50)
print("Extra Trees Regression Results")
print("=" * 50)

print("\nTrain Set:")
print(f"R²   : {r2_train:.4f}")
print(f"MAE  : {mae_train:.2f}")
print(f"MSE  : {mse_train:.2f}")
print(f"RMSE : {rmse_train:.2f}")

print("\nTest Set:")
print(f"R²   : {r2_test:.4f}")
print(f"MAE  : {mae_test:.2f}")
print(f"MSE  : {mse_test:.2f}")
print(f"RMSE : {rmse_test:.2f}")

print("\nR² Gap:")
print(f"Train R² - Test R² = {r2_gap:.4f}")

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1465431552.5532,4118095184943767040.0000,2029309041.2610,0.5565,0.7446,0.9498
1,1318342067.8265,3703197455936309760.0000,1924369365.7758,0.5649,0.7059,0.8606
2,1441227113.9088,4518596167765389824.0000,2125698983.3383,0.4706,0.7906,1.2009
3,1366007421.1189,3945931798974350336.0000,1986436960.7351,0.5424,0.7282,0.9171
4,1296629401.2120,3911351393489046016.0000,1977713678.3390,0.3668,0.7214,0.8700
5,1535100740.4192,4973560017274404864.0000,2230147981.0260,0.4929,0.6702,0.7289
6,1335885305.3268,3973947912654557696.0000,1993476338.6242,0.5340,0.6206,0.6703
7,1372345163.6613,3694617664121761280.0000,1922138825.4030,0.4657,0.7483,0.8986
8,1307979992.3990,3294637028125476352.0000,1815113502.8216,0.6199,0.6705,0.7810


C:\Users\Administrator\Desktop\projects\Projects\House Price Prediction\venv\Lib\site-packages\pycaret\internal\pycaret_experiment\pycaret_experiment.py:321: UserWarning: Variable: 'X_train' used to return the transformed values in PyCaret 2.x. From PyCaret 3.x, this will return the raw values. If you need the transformed values, call get_config with 'X_train_transformed' instead.
  warnings.warn(msg)  # print on screen


C:\Users\Administrator\Desktop\projects\Projects\House Price Prediction\venv\Lib\site-packages\pycaret\internal\pycaret_experiment\pycaret_experiment.py:321: UserWarning: Variable: 'y_train' used to return the transformed values in PyCaret 2.x. From PyCaret 3.x, this will return the raw values. If you need the transformed values, call get_config with 'y_train_transformed' instead.
  warnings.warn(msg)  # print on screen


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extra Trees Regressor,1358306431.2826,3808929748922541568.0000,1951647957.2204,0.5428,0.7342,1.5640


Extra Trees Regression Results

Train Set:
R²   : 0.5926
MAE  : 1255506918.42
MSE  : 3385720400831487488.00
RMSE : 1840032717.33

Test Set:
R²   : 0.5428
MAE  : 1358306431.28
MSE  : 3808929748922541568.00
RMSE : 1951647957.22

R² Gap:
Train R² - Test R² = 0.0498
